# Social Network Extraction on Google Colab

**Runtime → Change runtime type → A100 GPU** (or best available)

Passages should be pre-exported as JSONL files and uploaded to Google Drive.

See `scripts/hpc/export_passages.py` for the export step (run locally where ClickHouse is available).

## 0. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Install

In [ ]:
!pip install -q vllm
!pip install -q git+https://github.com/quadrismegistus/largeliterarymodels.git

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob

DRIVE_BASE = '/content/drive/MyDrive/social_network'
TEXT_DIR = os.path.join(DRIVE_BASE, 'texts')
OUTPUT_DIR = '/content/output'  # local SSD for speed, sync to Drive later
DRIVE_OUTPUT = os.path.join(DRIVE_BASE, 'output')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

n_texts = len(glob.glob(os.path.join(TEXT_DIR, '*.jsonl')))
n_done_drive = len(glob.glob(os.path.join(DRIVE_OUTPUT, '*.json')))
print(f'Texts available: {n_texts} JSONL files')
print(f'Already done (on Drive): {n_done_drive}')

if n_texts == 0:
    print(f'\nNo JSONL files found in {TEXT_DIR}.')
    print('Upload exported passages to Google Drive at: My Drive/social_network/texts/')
    print('See scripts/hpc/export_passages.py for the export step.')

## 3. Copy previous results from Drive (for skip-existing)

In [ ]:
# Copy any existing results from Drive to local so skip-existing works
existing = glob.glob(os.path.join(DRIVE_OUTPUT, '*.json'))
if existing:
    for f in existing:
        dst = os.path.join(OUTPUT_DIR, os.path.basename(f))
        if not os.path.exists(dst):
            !cp "{f}" "{dst}"
    print(f'Copied {len(existing)} existing results to local')
else:
    print('No existing results — starting fresh')

## 4. Start vLLM server

In [ ]:
import subprocess

MODEL = 'Qwen/Qwen3.6-27B'

proc = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL,
    '--port', '8000',
    '--host', '127.0.0.1',
    '--enable-prefix-caching',
    '--gpu-memory-utilization', '0.90',
    '--max-model-len', '32768',
    '--disable-log-requests',
], stdout=open('/content/vllm.log', 'w'), stderr=subprocess.STDOUT)
print(f'vLLM starting (pid {proc.pid}), logging to /content/vllm.log')

In [ ]:
import time, urllib.request

for i in range(120):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/health')
        print(f'vLLM ready after {i * 5}s!')
        break
    except:
        if i % 12 == 0:
            print(f'Waiting... ({i * 5}s)')
        time.sleep(5)
else:
    print('vLLM failed to start. Last 30 lines of log:')
    !tail -30 /content/vllm.log

## 5. Run batch

In [ ]:
WORKERS = 4

!python -m largeliterarymodels.cli run-batch-social-network 2>/dev/null || \
  python -c "
import subprocess, sys
# Find the batch script
import largeliterarymodels
import os
pkg = os.path.dirname(largeliterarymodels.__file__)
script = os.path.join(os.path.dirname(pkg), 'scripts', 'batch_social_network.py')
if not os.path.exists(script):
    # pip install puts it in site-packages, clone for scripts
    print('batch script not found at', script)
    print('Cloning repo for scripts...')
    os.system('git clone --depth 1 https://github.com/quadrismegistus/largeliterarymodels.git /content/llm_repo')
    script = '/content/llm_repo/scripts/batch_social_network.py'
print('Using:', script)
" 2>&1 | tail -5

# Clone repo for scripts if needed
import os
SCRIPT = '/content/llm_repo/scripts/batch_social_network.py'
if not os.path.exists(SCRIPT):
    !git clone --depth 1 https://github.com/quadrismegistus/largeliterarymodels.git /content/llm_repo

!python {SCRIPT} \
    --text-dir {TEXT_DIR} \
    --output-dir {OUTPUT_DIR} \
    --model vllm-qwen36 \
    --workers {WORKERS}

## 6. Sync results to Drive

Run this periodically (or after the batch finishes) to save results to Drive.
If the Colab session dies, results on local `/content/output/` are lost — but
anything synced to Drive persists.

In [ ]:
import shutil

local_results = glob.glob(os.path.join(OUTPUT_DIR, '*.json'))
synced = 0
for f in local_results:
    dst = os.path.join(DRIVE_OUTPUT, os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy2(f, dst)
        synced += 1

total = len(glob.glob(os.path.join(DRIVE_OUTPUT, '*.json')))
print(f'Synced {synced} new results to Drive ({total} total on Drive)')

## 7. Check progress

In [ ]:
import json

n_input = len(glob.glob(os.path.join(TEXT_DIR, '*.jsonl')))
n_done = len(glob.glob(os.path.join(OUTPUT_DIR, '*.json')))
n_drive = len(glob.glob(os.path.join(DRIVE_OUTPUT, '*.json')))
print(f'{n_done}/{n_input} done this session')
print(f'{n_drive}/{n_input} total on Drive')

if n_done > 0:
    latest = max(glob.glob(os.path.join(OUTPUT_DIR, '*.json')), key=os.path.getmtime)
    with open(latest) as f:
        d = json.load(f)
    m = d.get('metadata', {})
    print(f'\nLatest: {os.path.basename(latest)}')
    print(f'  Chars: {len(d.get("characters", []))}, Events: {len(d.get("events", []))}')
    print(f'  Elapsed: {m.get("elapsed_seconds", 0)/60:.1f} min')

## 8. Cleanup

In [ ]:
# Stop vLLM
proc.terminate()
print('vLLM stopped')

# Final sync to Drive
local_results = glob.glob(os.path.join(OUTPUT_DIR, '*.json'))
for f in local_results:
    dst = os.path.join(DRIVE_OUTPUT, os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy2(f, dst)

total = len(glob.glob(os.path.join(DRIVE_OUTPUT, '*.json')))
print(f'{total} results saved on Drive')